In [15]:
with open('external_urls_3.txt', 'r', encoding='utf-8') as f:
    urls = f.readlines()

In [16]:
unique_urls = set(urls)

In [17]:
len(unique_urls)

6035

In [11]:
def load_top_cn_sites(filename):
    with open(filename, 'r') as f:
        return set(line.strip() for line in f)

def is_top_cn_site(url, top_sites):
    domain = urlparse(url).netloc
    domain = ".".join(domain.split('.')[-2:])
    return domain in top_sites
top_sites = load_top_cn_sites('top_sites_cn.txt')


In [19]:
async def is_ai_related(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            html = await response.text()

    soup = BeautifulSoup(html, 'html.parser')
    meta_tags = soup.find_all('meta')
    ai_related_keywords = ['ai', 'artificial intelligence', 'machine learning', 'deep learning', 'neural network']

    for tag in meta_tags:
        if 'name' in tag.attrs and 'content' in tag.attrs:
            if tag.attrs['name'].lower() in ['description', 'keywords']:
                for keyword in ai_related_keywords:
                    if keyword in tag.attrs['content'].lower():
                        return True

    return False

In [20]:
def get_url_netloc(url):
    return urlparse(url).netloc

In [21]:
for url in unique_urls:
    if not is_top_cn_site(url, top_sites):
        netloc = get_url_netloc(url)
        if not await is_ai_related(url):
            print(url)

/opt/homebrew/Cellar/python@3.11/3.11.4_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/ast.py:50: RuntimeWarning: coroutine 'main' was never awaited
  return compile(source, filename, mode, flags,


ServerDisconnectedError: Server disconnected

In [14]:
import aiohttp
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin, urlunparse

import asyncio
from fake_useragent import UserAgent


class WebsiteCrawler:

    def __init__(self, base_url, max_depth):
        self.base_url = base_url
        self.base_domain = urlparse(base_url).netloc
        self.max_depth = max_depth
        self.external_urls = set()
        self.queue = asyncio.Queue()
        self.visited = set()
        self.ua = UserAgent()
        self.save_path = "external_urls_3.txt"
        with open(self.save_path, "r", encoding='utf-8') as f:
            for line in f.readlines():
                self.external_urls.add(line.strip())

    async def write_to_file(self, url):
        with open(self.save_path, "a", encoding='utf-8') as f:
            f.write(url + "\n")

    async def crawl(self, url, depth):
        if depth > self.max_depth or url in self.visited:
            print(f"Skipping {url}")
            return

        self.visited.add(url)

        async with aiohttp.ClientSession(headers={'User-Agent': self.ua.random}) as session:
            try:
                html = await self.fetch(session, url)
            except Exception as e:
                print(f"Error fetching {url}: {e}")
                return

        soup = BeautifulSoup(html, "html.parser")

        for a_tag in soup.findAll("a"):
            href = a_tag.attrs.get("href")
            if href == "" or href is None:
                continue
            href = urljoin(url, href)
            parsed_href = urlparse(href)
            href_contains_domain = self.base_domain in parsed_href.netloc
            base_url_parts = parsed_href._replace(query="", fragment="")
            base_url = urlunparse(base_url_parts)
            is_top_cn_site = base_url in top_sites
            if is_top_cn_site:
                continue
            if not href_contains_domain and not parsed_href.netloc == "" and parsed_href.scheme in ["http", "https"]:
                if base_url not in self.external_urls:
                    print(f"Found new external link: {href}")
                    self.external_urls.add(base_url)
                    await self.write_to_file(base_url)
            elif href_contains_domain and parsed_href.scheme in ["http", "https"]:
                await self.queue.put((base_url, depth + 1))

    async def fetch(self, session, url):
        async with session.get(url) as response:
            return await response.text()

    async def run(self):
        await self.queue.put((self.base_url, 1))
        while not self.queue.empty():
            url, depth = await self.queue.get()
            print(f"Fetching {url} at depth {depth}")
            await self.crawl(url, depth)
        print("\n".join(self.external_urls))


async def main():
    with open("external_urls_2.txt", "r", encoding='utf-8') as f:
        start_urls = f.readlines()
    for start_url in start_urls:
        crawler = WebsiteCrawler(start_url.strip(), 3)
        await crawler.run()


loop = asyncio.get_event_loop()
loop.run_until_complete(main())


RuntimeError: This event loop is already running